In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression,LogisticRegression,Ridge
from sklearn.metrics import (
    mean_squared_error, r2_score,
    confusion_matrix, classification_report,
    roc_curve, roc_auc_score,
    precision_score, recall_score, f1_score
)
#Load the cleaned.csv dataset
df=pd.read_csv("/content/cleaned_data.csv")
print(f"Cleaned data has been loaded Successfully with shape:{df.shape}")

# Feature matrix: drop the regression target and any pure identifier columns
# that carry no generalizable signal (high-cardinality unique IDs).
id_like_cols = ["Item_Identifier"]  # 1559 unique values -> not a usable feature
X = df.drop(columns=["Item_Outlet_Sales"] + id_like_cols)
print(X)
#Regression label y_reg: one continuous numeric column (e.g., price, income).
y_reg=df['Item_Outlet_Sales']

#Classification label
y_clf = (y_reg > y_reg.median()).astype(int)

print(f"\ny_reg = 'Item_Outlet_Sales' (continuous). Median = {y_reg.median():.2f}")
print("y_clf = 1 if Item_Outlet_Sales > median, else 0 (binarized at the median)")
print("\nBefore any resampling — full-dataset class balance of y_clf:")
print(y_clf.value_counts())
print(y_clf.value_counts(normalize=True).rename("proportion"))

Cleaned data has been loaded Successfully with shape:(8523, 13)
      Item_Weight Item_Fat_Content  Item_Visibility              Item_Type  \
0           9.300          Low Fat         0.016047                  Dairy   
1           5.920          Regular         0.019278            Soft Drinks   
2          17.500          Low Fat         0.016760                   Meat   
3          19.200          Regular         0.000000  Fruits and Vegetables   
4           8.930          Low Fat         0.000000              Household   
...           ...              ...              ...                    ...   
8518        6.865          Low Fat         0.056783            Snack Foods   
8519        8.380          Regular         0.046982           Baking Goods   
8520       10.600          Low Fat         0.035186     Health and Hygiene   
8521        7.210          Regular         0.145221            Snack Foods   
8522       14.800          Low Fat         0.044878            Soft Drinks   


In [ ]:
#Encode categorical column

cat_cols= X.select_dtypes(include=["object","string"]).columns.tolist()
print(f"Categorical Columns identified are:{cat_cols}")

# --- Ordinal encodings (natural order exists) ---

# Outlet_Size has a natural small < medium < high ordering
outlet_size_map={"Small":0,"Medium":1,"High":2}
X['Outlet_Size']=X['Outlet_Size'].map(outlet_size_map)

#Outlet_Location_Type: Tier 1 (largest/most developed) > Tier 2 > Tier 3 in the
# standard Indian retail classification, so tier number reflects an ordered
# decrease in market development. We map Tier 1 -> 2, Tier 2 -> 1, Tier 3 -> 0
# so that larger encoded values consistently correspond to "more developed".
outlet_tier_map = {"Tier 3": 0, "Tier 2": 1, "Tier 1": 2}
X["Outlet_Location_Type"] = X["Outlet_Location_Type"].map(outlet_tier_map)

# --- One-hot encodings (no natural order) ---
nominal_cols = ["Item_Fat_Content", "Item_Type", "Outlet_Identifier", "Outlet_Type"]
print(f"\nApplied ONE-HOT encoding (drop_first=True) to: {nominal_cols}")
print("Reason: these are category labels with no inherent ranking (e.g., 'Dairy' is not")
print("greater or less than 'Snack Foods'). Label-encoding them as 0,1,2... would invent a")
print("false ordinal relationship that the model would wrongly treat as meaningful magnitude.")

X = pd.get_dummies(X, columns=nominal_cols, drop_first=True)

print(f"\nFeature matrix X shape (after encoding): {X.shape}")


Categorical Columns identified are:['Item_Fat_Content', 'Item_Type', 'Outlet_Identifier', 'Outlet_Size', 'Outlet_Location_Type', 'Outlet_Type']

Applied ONE-HOT encoding (drop_first=True) to: ['Item_Fat_Content', 'Item_Type', 'Outlet_Identifier', 'Outlet_Type']
Reason: these are category labels with no inherent ranking (e.g., 'Dairy' is not
greater or less than 'Snack Foods'). Label-encoding them as 0,1,2... would invent a
false ordinal relationship that the model would wrongly treat as meaningful magnitude.

Feature matrix X shape (after encoding): (8523, 35)


In [ ]:
#Leak-free train-test split and scaling


X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42
)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
scaler = StandardScaler()
scaler.fit(X_train)  # fit ONLY on training data
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)


print("Scaler fit on X_train only, then used to transform both X_train and X_test.")
print("(Fitting on the full dataset would leak test-set mean/variance into training —")
print(" see README for the full explanation.)")

feature_names = X.columns.tolist()


X_train: (6818, 35), X_test: (1705, 35)
Scaler fit on X_train only, then used to transform both X_train and X_test.
(Fitting on the full dataset would leak test-set mean/variance into training —
 see README for the full explanation.)


In [ ]:
#Regression model — Linear Regression:

lin_reg=LinearRegression()
lin_reg.fit(X_train_scaled,y_reg_train)
y_pred_reg=lin_reg.predict(X_test_scaled)
mse_lin=mean_squared_error(y_reg_test, y_pred_reg)
r2_lin = r2_score(y_reg_test, y_pred_reg)

print(f"Linear Regression -> MSE: {mse_lin:.4f}, R^2: {r2_lin:.4f}")

coef_df = pd.DataFrame({"feature": feature_names, "coefficient": lin_reg.coef_})
coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
coef_df_sorted = coef_df.sort_values("abs_coefficient", ascending=False)

print("\nLinear Regression coefficients (sorted by |coefficient|):")
print(coef_df_sorted.drop(columns="abs_coefficient").to_string(index=False))

top3 = coef_df_sorted.head(3)
print("\nTop 3 features by absolute coefficient magnitude:")
print(top3.drop(columns="abs_coefficient").to_string(index=False))

Linear Regression -> MSE: 1143541.2997, R^2: 0.5793

Linear Regression coefficients (sorted by |coefficient|):
                        feature  coefficient
                       Item_MRP   979.457812
  Outlet_Type_Supermarket Type1   478.801193
       Outlet_Identifier_OUT027   464.774601
  Outlet_Type_Supermarket Type3   464.774601
                    Outlet_Size   318.618352
           Outlet_Location_Type   234.626028
       Outlet_Identifier_OUT035   230.490626
       Outlet_Identifier_OUT017   208.564292
       Outlet_Identifier_OUT018   175.884776
  Outlet_Type_Supermarket Type2   175.884776
       Outlet_Identifier_OUT045   172.573036
       Outlet_Identifier_OUT019  -116.005909
       Outlet_Identifier_OUT046    98.537961
       Outlet_Identifier_OUT013    31.365215
                Item_Type_Dairy   -24.093111
                Item_Visibility   -23.890325
       Item_Fat_Content_Regular    21.491108
              Item_Type_Seafood    19.187331
                     Outlet_Age   

In [ ]:
# Ridge Regression
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_reg_train)
y_pred_ridge = ridge.predict(X_test_scaled)

mse_ridge = mean_squared_error(y_reg_test, y_pred_ridge)
r2_ridge = r2_score(y_reg_test, y_pred_ridge)

print(f"\nRidge Regression (alpha=1.0) -> MSE: {mse_ridge:.4f}, R^2: {r2_ridge:.4f}")

comparison_reg = pd.DataFrame({
    "Model": ["Linear Regression", "Ridge (alpha=1.0)"],
    "MSE": [mse_lin, mse_ridge],
    "R2": [r2_lin, r2_ridge],
})
print("\nComparison table:")
print(comparison_reg.to_string(index=False))


Ridge Regression (alpha=1.0) -> MSE: 1143527.3222, R^2: 0.5793

Comparison table:
            Model          MSE       R2
Linear Regression 1.143541e+06 0.579266
Ridge (alpha=1.0) 1.143527e+06 0.579272


In [ ]:
#Classification model — Logistic Regression:
train_counts = y_clf_train.value_counts()
train_props = y_clf_train.value_counts(normalize=True)
print("y_clf_train class counts (BEFORE any imbalance handling):")
print(train_counts)
print(train_props.rename("proportion"))

minority_prop = train_props.min()
imbalanced = minority_prop < 0.35
print(f"\nMinority class proportion in training set: {minority_prop:.4f}")
print(f"Imbalance check (<35% threshold): {'IMBALANCED' if imbalanced else 'balanced enough — threshold not triggered'}")

# The task requires demonstrating imbalance handling. Even though the median-split
# label lands close to 50/50 for this dataset, class_weight='balanced' is applied
# as the chosen technique (option (b)) so the mechanism and before/after weighting
# is fully demonstrated. SMOTE (option (a)) was not available in this environment
# (no package index access to install imbalanced-learn), so class_weight='balanced'
# is used instead — see README for the justification.
n_samples = len(y_clf_train)
n_classes = y_clf_train.nunique()
class_weights = {
    cls: n_samples / (n_classes * count) for cls, count in train_counts.items()
}
print(f"\nChosen imbalance-handling technique: class_weight='balanced' in LogisticRegression")
print(f"Resulting per-class weights (AFTER handling, weights not resampled counts): {class_weights}")
print("Note: class counts themselves are unchanged (no rows added/removed) — 'balanced'")
print("reweights the loss function so the minority class contributes proportionally more.")

logreg = LogisticRegression(max_iter=1000, class_weight="balanced", C=1.0, random_state=42)
logreg.fit(X_train_scaled, y_clf_train)

y_pred_clf = logreg.predict(X_test_scaled)
y_proba_clf = logreg.predict_proba(X_test_scaled)[:, 1]

cm = confusion_matrix(y_clf_test, y_pred_clf)
print("\nConfusion matrix (rows=actual, cols=predicted), classes=[0,1]:")
print(cm)

report = classification_report(y_clf_test, y_pred_clf)
print("\nClassification report:")
print(report)

auc = roc_auc_score(y_clf_test, y_proba_clf)
fpr, tpr, thresholds_roc = roc_curve(y_clf_test, y_proba_clf)
print(f"AUC (C=1.0, balanced): {auc:.4f}")

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC curve (AUC = {auc:.4f})")
plt.plot([0, 1], [0, 1], color="navy", lw=1, linestyle="--", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Logistic Regression (C=1.0, class_weight='balanced')")
plt.annotate(f"AUC = {auc:.4f}", xy=(0.55, 0.15), fontsize=12,
             bbox=dict(boxstyle="round", fc="white", ec="gray"))
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("roc_curve.png", dpi=150)
plt.close()
print("Saved ROC curve plot -> roc_curve.png")

y_clf_train class counts (BEFORE any imbalance handling):
Item_Outlet_Sales
1    3424
0    3394
Name: count, dtype: int64
Item_Outlet_Sales
1    0.5022
0    0.4978
Name: proportion, dtype: float64

Minority class proportion in training set: 0.4978
Imbalance check (<35% threshold): balanced enough — threshold not triggered

Chosen imbalance-handling technique: class_weight='balanced' in LogisticRegression
Resulting per-class weights (AFTER handling, weights not resampled counts): {1: 0.9956191588785047, 0: 1.0044195639363582}
Note: class counts themselves are unchanged (no rows added/removed) — 'balanced'
reweights the loss function so the minority class contributes proportionally more.

Confusion matrix (rows=actual, cols=predicted), classes=[0,1]:
[[721 150]
 [177 657]]

Classification report:
              precision    recall  f1-score   support

           0       0.80      0.83      0.82       871
           1       0.81      0.79      0.80       834

    accuracy                  

In [ ]:
#Decision-threshold sensitivity
thresholds = [0.30, 0.40, 0.50, 0.60, 0.70]
threshold_rows = []
for t in thresholds:
    preds_t = (y_proba_clf >= t).astype(int)
    p = precision_score(y_clf_test, preds_t)
    r = recall_score(y_clf_test, preds_t)
    f1 = f1_score(y_clf_test, preds_t)
    threshold_rows.append({"Threshold": t, "Precision": p, "Recall": r, "F1": f1})

threshold_df = pd.DataFrame(threshold_rows)
print(threshold_df.to_string(index=False))

best_f1_row = threshold_df.loc[threshold_df["F1"].idxmax()]
print(f"\nThreshold that maximizes F1: {best_f1_row['Threshold']:.2f} (F1 = {best_f1_row['F1']:.4f})")


 Threshold  Precision   Recall       F1
       0.3   0.731614 0.918465 0.814460
       0.4   0.780303 0.864508 0.820250
       0.5   0.814126 0.787770 0.800731
       0.6   0.840621 0.714628 0.772521
       0.7   0.858347 0.610312 0.713385

Threshold that maximizes F1: 0.40 (F1 = 0.8203)


In [ ]:
#Regularization Experiment

logreg_strong_reg = LogisticRegression(max_iter=1000, class_weight="balanced", C=0.01, random_state=42)
logreg_strong_reg.fit(X_train_scaled, y_clf_train)

y_pred_c001 = logreg_strong_reg.predict(X_test_scaled)
y_proba_c001 = logreg_strong_reg.predict_proba(X_test_scaled)[:, 1]

precision_c1 = precision_score(y_clf_test, y_pred_clf)
recall_c1 = recall_score(y_clf_test, y_pred_clf)
auc_c1 = auc

precision_c001 = precision_score(y_clf_test, y_pred_c001)
recall_c001 = recall_score(y_clf_test, y_pred_c001)
auc_c001 = roc_auc_score(y_clf_test, y_proba_c001)

reg_comparison = pd.DataFrame({
    "Model": ["Logistic Regression (C=1.0)", "Logistic Regression (C=0.01)"],
    "Precision": [precision_c1, precision_c001],
    "Recall": [recall_c1, recall_c001],
    "AUC": [auc_c1, auc_c001],
})
print(reg_comparison.to_string(index=False))


                       Model  Precision   Recall      AUC
 Logistic Regression (C=1.0)   0.814126 0.787770 0.895086
Logistic Regression (C=0.01)   0.809170 0.782974 0.892324


In [ ]:
#Bootstrap confidence interval for AUC difference
n_bootstrap = 500
y_clf_test_arr = y_clf_test.to_numpy()
auc_diffs = np.zeros(n_bootstrap)

rng = np.random.RandomState(42)
for i in range(n_bootstrap):
    idx = rng.choice(len(y_clf_test_arr), size=len(y_clf_test_arr), replace=True)
    y_sample = y_clf_test_arr[idx]
    # Skip degenerate samples where only one class is present (AUC undefined)
    if len(np.unique(y_sample)) < 2:
        auc_diffs[i] = np.nan
        continue
    proba_c1_sample = y_proba_clf[idx]
    proba_c001_sample = y_proba_c001[idx]
    auc_c1_sample = roc_auc_score(y_sample, proba_c1_sample)
    auc_c001_sample = roc_auc_score(y_sample, proba_c001_sample)
    auc_diffs[i] = auc_c1_sample - auc_c001_sample

valid_diffs = auc_diffs[~np.isnan(auc_diffs)]
mean_diff = np.mean(valid_diffs)
ci_lower = np.percentile(valid_diffs, 2.5)
ci_upper = np.percentile(valid_diffs, 97.5)

print(f"Bootstrap iterations used: {len(valid_diffs)} / {n_bootstrap}")
print(f"Mean AUC difference (C=1.0 - C=0.01): {mean_diff:.4f}")
print(f"95% CI: [{ci_lower:.4f}, {ci_upper:.4f}]")
excludes_zero = (ci_lower > 0) or (ci_upper < 0)
print(f"95% CI excludes zero: {excludes_zero}")


Bootstrap iterations used: 500 / 500
Mean AUC difference (C=1.0 - C=0.01): 0.0028
95% CI: [0.0014, 0.0044]
95% CI excludes zero: True


In [ ]:
results = {
    "mse_lin": mse_lin, "r2_lin": r2_lin,
    "mse_ridge": mse_ridge, "r2_ridge": r2_ridge,
    "top3_features": top3["feature"].tolist(),
    "top3_coefs": top3["coefficient"].tolist(),
    "auc_c1": auc_c1, "auc_c001": auc_c001,
    "precision_c1": precision_c1, "recall_c1": recall_c1,
    "precision_c001": precision_c001, "recall_c001": recall_c001,
    "best_f1_threshold": float(best_f1_row["Threshold"]),
    "best_f1_value": float(best_f1_row["F1"]),
    "mean_auc_diff": mean_diff,
    "ci_lower": ci_lower,
    "ci_upper": ci_upper,
    "excludes_zero": bool(excludes_zero),
    "train_class_counts": train_counts.to_dict(),
    "minority_prop": float(minority_prop),
}

import json
with open("results.json", "w") as f:
    json.dump(results, f, indent=2, default=str)

print("\nAll results saved to results.json")
print("\nDONE.")



All results saved to results.json

DONE.
